# Занятие 2, демо 2. Delta rule - это не нормализация

Обе конструкции «чинят» накопление в состоянии, и на слух разница кажется
косметической. Посмотрим, где они расходятся.

In [ ]:
import torch

torch.set_num_threads(1)

"""Конечная память: запись, интерференция, перезапись.

Материал занятия 2. Additive-запись и delta rule - два правила **записи** в
одно и то же состояние. Нормализация устроена иначе: состояние копится тем же
additive-правилом, но рядом копится масса ядра, и меняется способ **чтения**.
"""
import torch


def additive_write(state, k, v):
    """Слагаемое: S <- S + v k^T. Прошлое не трогается."""
    return state + v.unsqueeze(-1) * k.unsqueeze(-2)


def delta_write(state, k, v, beta=1.0):
    """Delta rule: сначала читается то, что уже лежит по ключу, потом правится.

    S <- S + beta * (v - S k) k^T. При beta = 1 и единичном ключе чтение по
    этому ключу становится ровно v.
    """
    current = (state @ k.unsqueeze(-1)).squeeze(-1)
    return state + beta * (v - current).unsqueeze(-1) * k.unsqueeze(-2)


def read(state, q):
    """Чтение: y = S q."""
    return (state @ q.unsqueeze(-1)).squeeze(-1)


def unit(*values, dtype=torch.float64):
    """Единичный вектор заданного направления."""
    t = torch.tensor(values, dtype=dtype)
    return t / t.norm()


def rotate(vector, angle):
    """Поворот двумерного вектора на угол в радианах."""
    c, s = torch.cos(torch.tensor(angle, dtype=vector.dtype)), torch.sin(
        torch.tensor(angle, dtype=vector.dtype))
    return torch.stack([c * vector[0] - s * vector[1],
                        s * vector[0] + c * vector[1]])


def zero_state(d_k, d_v, dtype=torch.float64):
    """Состояние имеет форму d_v x d_k: та же, что на семинаре."""
    return torch.zeros(d_v, d_k, dtype=dtype)


def normalized_additive(keys, values, q, eps=0.0):
    """Нормированное additive linear attention, в той же форме, что и delta.

    Состояние копится обычным additive-правилом, рядом копится масса ядра
    z = sum k_i. Меняется только чтение: y = S q / (z^T q). Возвращает
    (состояние, масса, ответ).

    Так видно, что это не альтернативное правило записи: в S лежит ровно то же,
    что и без нормализации.
    """
    state = zero_state(q.shape[-1], values[0].shape[-1], dtype=values[0].dtype)
    mass = torch.zeros_like(q)
    for k, v in zip(keys, values):
        state = additive_write(state, k, v)
        mass = mass + k
    return state, mass, read(state, q) / (q @ mass + eps)


def delta_sequence(keys, values, q, d_k, d_v, beta=1.0):
    """Последовательная запись delta rule и чтение по q."""
    state = zero_state(d_k, d_v, dtype=values[0].dtype)
    for k, v in zip(keys, values):
        state = delta_write(state, k, v, beta=beta)
    return state, read(state, q)


def random_memory(n, d_k, d_v, generator):
    """n случайных ассоциаций: единичные ключи, гауссовы значения."""
    keys = []
    for _ in range(n):
        key = torch.randn(d_k, generator=generator, dtype=torch.float64)
        keys.append(key / key.norm())
    values = [torch.randn(d_v, generator=generator, dtype=torch.float64)
              for _ in range(n)]
    state = zero_state(d_k, d_v)
    for key, value in zip(keys, values):
        state = additive_write(state, key, value)
    return state, keys, values


def interference_rms(n, d_k, d_v, trials, seed):
    """Среднеквадратичная ошибка чтения первой пары по многим реализациям.

    Одна реализация ничего не показывает: ошибка случайна и от n к n может
    как расти, так и падать. Растёт именно типичная величина.
    """
    generator = torch.Generator().manual_seed(seed)
    total = 0.0
    for _ in range(trials):
        state, keys, values = random_memory(n, d_k, d_v, generator)
        total += float((read(state, keys[0]) - values[0]).norm()) ** 2
    return (total / trials) ** 0.5

## Кадр первый: один ключ

Скалярный случай, чтобы всё было видно: $k_1=k_2=q=1$, $v_1=1$, $v_2=3$.

**Нормализация** копит состояние обычной additive-записью, а рядом - массу ядра,
и меняет **чтение**: $y=Sq/(z^\top q)$.

**Delta rule** меняет саму **запись**: перед ней читается то, что уже лежит по
ключу, и пишется поправка $v-Sk$.

In [ ]:
one = torch.tensor([1.0], dtype=torch.float64)
keys = [one, one]
values = [torch.tensor([1.0], dtype=torch.float64),
          torch.tensor([3.0], dtype=torch.float64)]

state, mass, y_norm = normalized_additive(keys, values, one)
state_delta, y_delta = delta_sequence(keys, values, one, d_k=1, d_v=1)

print(f"нормализация: S = {float(state):.1f}, масса = {float(mass):.1f},"
      f" ответ = {float(y_norm):.1f}")
print(f"delta rule:   S = {float(state_delta):.1f},"
      f" ответ = {float(y_delta):.1f}")
print()
print(f"последнее записанное значение: {float(values[-1]):.1f}")

Соблазнительный вывод: «нормализация даёт среднее, delta - последнее
записанное». Он неверен, и следующий кадр это показывает.

Оба утверждения держатся на том, что ключи **одинаковы** и единичны. В общем
случае нормализация даёт нормированную взвешенную сумму с весами
$w_i=q^\top k_i$ - средним она становится только при равных весах. А delta
возвращает ровно $v$ лишь при $\beta\lVert k\rVert^2=1$ и чтении тем же ключом.

## Кадр второй: ключи разные

$k_1=(1,0)$, $k_2=(0.6,0.8)$ - оба единичные, но уже не совпадают.
Записываем $v_1=1$ и $v_2=3$, читаем по $k_1$. И меняем порядок записей.

In [ ]:
k1 = torch.tensor([1.0, 0.0], dtype=torch.float64)
k2 = torch.tensor([0.6, 0.8], dtype=torch.float64)
v1 = torch.tensor([1.0], dtype=torch.float64)
v2 = torch.tensor([3.0], dtype=torch.float64)

print(f"{'порядок записей':<18} {'нормализация':>14} {'delta rule':>12}")
for label, order_k, order_v in (("k1, потом k2", [k1, k2], [v1, v2]),
                                ("k2, потом k1", [k2, k1], [v2, v1])):
    _, _, y_n = normalized_additive(order_k, order_v, k1)
    _, y_d = delta_sequence(order_k, order_v, k1, d_k=2, d_v=1)
    print(f"{label:<18} {float(y_n):>14.2f} {float(y_d):>12.2f}")

print()
print("Читаем по k1. Последнее записанное в первой строке - 3.00.")

## Что видно

**Нормализация не зависит от порядка записей.** Она делит накопленную сумму на
накопленную массу, а сумма от порядка не зависит.

**Delta rule зависит.** Она читает состояние перед записью, а состояние разное
в зависимости от того, что записали раньше.

И в первой строке delta даёт $2.44$, а не последнее записанное $3$. «Последнее
значение» получалось только там, где ключи совпадали.

Что delta делает на самом деле: при $\beta=1$ и единичном ключе

$$
S^+=S(I-kk^\top)+vk^\top
$$

то есть заменяется **та компонента состояния, которая читается этим ключом** -
после записи $Sk=v$. Остальное состояние остаётся, и чтение по
неортогональному ключу тоже изменится.

Нормализация ничего подобного не делает: она не вычисляет ошибку предсказания
$v-\hat v$ и состояние не трогает вовсе. Это не «тот же приём другими словами».